Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.messages import HumanMessage, AIMessage, SystemMessage

Text length Based Summarization

In [ ]:

agent = create_agent(
    model="gpt-5.4",
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=('messages', 5),
            keep=('messages',4)
        ),
        #HumanInTheLoopMiddleware(...)
    ],
)

In [13]:
#Run with thread ID
config = { "configurable": { "thread_id": "user-1234" }}

In [14]:
questions =[
    "What is the capital of France?",
    "Who is the president of the United States?",
    "What is the largest mammal?",
    "How many planets are in our solar system?",
    "What is the boiling point of water?",
    "Who wrote 'To Kill a Mockingbird'?"

]

In [15]:
for q in questions:
    response = agent.invoke({'messages': [HumanMessage(content=q)]}, config=config)
    print(f"Messages: {q}\nA: {response}\n")
    print(f"Messages: {q}\nA: {len(response['messages'])}\n")

Messages: What is the capital of France?
A: {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='6966e481-76f1-421c-9423-2e4140c378c5'), AIMessage(content='Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 13, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DmTuPQN8JS9ncRSLBpogJgcVfIDqU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e8aea-802f-7043-904e-5228f970cb73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 5, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 

In [16]:
from langchain_core.tools import tool


@tool
def search_hotels(city: str) -> str:
    """
    Search hotels in a city.
    Returns hotel recommendations.
    """
    return f"""
Hotels in {city}:

1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi
"""

Token Based Summarization

In [17]:
#Token Based Summarization

agent = create_agent(
    model="gpt-5.4",
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=('tokens', 550),
            keep=('tokens',200)
        )
        #HumanInTheLoopMiddleware(...)
    ],
)

In [18]:
#Run with thread ID
config = { "configurable": { "thread_id": "user-1235" }}

In [21]:
# Token counter (approximate)

import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

In [23]:
# Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(" ".join([msg.content for msg in response["messages"]]))

    print(
        f"{city}: ~{tokens} tokens, "
        f"{len(response['messages'])} messages"
    )

    print(f"{response['messages']}")

Paris: ~731 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='2d327100-7f0b-46f2-b41f-c70ec96f7a5a'), AIMessage(content='Sure — I can help with that.\n\nTo find the best hotels in Paris, please share any of these details you care about:\n\n- Check-in and check-out dates\n- Number of guests / rooms\n- Budget range per night\n- Preferred area: e.g. Eiffel Tower, Louvre, Champs-Élysées, Le Marais, Montmartre, Latin Quarter\n- Hotel style: luxury, boutique, budget, family-friendly, business, etc.\n- Any must-haves: breakfast, parking, pool, air conditioning, pet-friendly, free cancellation\n\nIf you want, I can also give you a quick shortlist right now by category, such as:\n\n- Luxury hotels in Paris\n- Budget-friendly hotels in central Paris\n- Romantic hotels\n- Family-friendly hotels\n- Hotels near the Eiffel Tower\n- Hotels near Disneyland Paris\n\nTell me your preferences and I’ll narrow it down.', additional_kwargs={'ref

In [28]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [29]:
agent = create_agent(
    model="gpt-5.4",
    tools=[read_email_tool, send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email_tool": True,  # All decisions (approve, edit, reject, respond) allowed
                "send_email_tool":{"allowed_decisions": ["approve","edit", "reject"]} # No editing allowed
                 #"execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                 #"read_data": False, # Safe operation, no approval needed
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),
)

In [37]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='47ff30fc-a56a-4858-bb68-fb0fe06e152b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 178, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DmUQ1tOJLtQfwUJ3kjtCa8SzK5fSk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8b08-69ab-7d90-a2e8-9dfaa50a622c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_2YmzCRnuJ9gqwbisufpe6R5e'

In [38]:
# Step 2: Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Sent.


In [39]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='47ff30fc-a56a-4858-bb68-fb0fe06e152b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 178, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DmUQ1tOJLtQfwUJ3kjtCa8SzK5fSk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8b08-69ab-7d90-a2e8-9dfaa50a622c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_2YmzCRnuJ9gqwbisufpe6R5e'